# **Hospital Readmission Prediction & Patient Risk Intelligence System**

### 1. Import Libraries and Loading Dataset

In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Healthforcast-Model/Diabetic_Cleaned_Data.csv")

Feature Engineering:
Converting raw healthcare data into features that machine learning models can understand and learn from.

### 2. Check Dataset Shape

In [ ]:
print(df.shape)

(101763, 46)


### 3. Create Targate Variable

In [ ]:
df['readmitted'] = df['readmitted'].replace({
    'NO':0,
    '>30':0,
    '<30':1
})

For Below purpose we are Doing above steps

The Objective of this Project is:

Predict whether a patient will return to the hospital within 30 days.

This is a binary classification problem.

### 4. Verify Targate Distribution

In [4]:
df['readmitted'].value_counts()

,count
readmitted,
0,90406
1,11357


### 5. Handeling Age Column

| Age Group | Numerical Value |
| --------- | --------------- |
| [0-10)    | 5               |
| [10-20)   | 15              |
| [20-30)   | 25              |


In [5]:
age_map = {
    '[0-10)':5,
    '[10-20)':15,
    '[20-30)':25,
    '[30-40)':35,
    '[40-50)':45,
    '[50-60)':55,
    '[60-70)':65,
    '[70-80)':75,
    '[80-90)':85,
    '[90-100)':95
}

df['age'] = df['age'].map(age_map)

### 6. Handle A1Cresult

Higher HbA1c means poorer diabetes control.

Higher values should correspond to higher risk.

This is an ordinal feature.

In [7]:
a1c_map = {
    'None':0,
    'Norm':1,
    '>7':2,
    '>8':3
}

df['A1Cresult'] = df['A1Cresult'].map(a1c_map)

### 7. Handle max_glu_serum

Higher glucose values indicate more severe disease.

In [8]:
glu_map = {
    'None':0,
    'Norm':1,
    '>200':2,
    '>300':3
}

df['max_glu_serum'] = df['max_glu_serum'].map(glu_map)

### 8. Handle Binary Features

diabetesMed and change are the most obvious Yes/No columns.

so Binary features should become numerical.

In [11]:
binary_map = {
    'No':0,
    'Yes':1
}

df['diabetesMed'] = df['diabetesMed'].map(binary_map)

df['change'] = df['change'].map(binary_map)

### 9. Create Total Visit Feature

Patients with many previous hospital visits usually have:

chronic conditions
severe disease
higher readmission risk

In [12]:
df['total_visits'] = (
    df['number_outpatient']
    + df['number_emergency']
    + df['number_inpatient']
)

### 10. Create Medication Burden Feature

Patients taking many medications often have:

1. multiple diseases
2. severe conditions

In [15]:
df['high_medication'] = (
    df['num_medications'] > 20
).astype(int)

### 11. Create Procedure Burden Feature

Patients requiring many procedures are often more severe cases.

In [16]:
df['total_procedures'] = (
    df['num_lab_procedures']
    + df['num_procedures']
)

### 12. Handle Diagnosis Columns

Group diagnoses into broader disease categories.

| ICD Code Range | Disease Category        |
| -------------- | ----------------------- |
| 250.xx         | Diabetes                |
| 390-459        | Circulatory Disease     |
| 460-519        | Respiratory Disease     |
| 520-579        | Digestive Disease       |
| 580-629        | Genitourinary Disease   |
| 710-739        | Musculoskeletal Disease |
| 800-999        | Injury/Poisoning        |
| Others         | Other                   |


In [17]:
def categorize_diagnosis(code):

    try:
        code = float(code)

        if 390 <= code <= 459:
            return "Circulatory"

        elif 460 <= code <= 519:
            return "Respiratory"

        elif 520 <= code <= 579:
            return "Digestive"

        elif 580 <= code <= 629:
            return "Genitourinary"

        elif 710 <= code <= 739:
            return "Musculoskeletal"

        elif code == 250:
            return "Diabetes"

        elif 800 <= code <= 999:
            return "Injury"

        else:
            return "Other"

    except:
        return "Other"

In [18]:
df['diag_1_cat'] = df['diag_1'].apply(categorize_diagnosis)

df['diag_2_cat'] = df['diag_2'].apply(categorize_diagnosis)

df['diag_3_cat'] = df['diag_3'].apply(categorize_diagnosis)

In [19]:
df.drop(
    columns=[
        'diag_1',
        'diag_2',
        'diag_3'
    ],
    inplace=True
)

In [20]:
df = pd.get_dummies(
    df,
    columns=[
        'diag_1_cat',
        'diag_2_cat',
        'diag_3_cat'
    ]
)

### 13. Identify Remaining Categorical Columns

In [22]:
categorical_cols = df.select_dtypes(include='object').columns

print(categorical_cols)

Index(['race', 'gender', 'medical_specialty', 'metformin', 'repaglinide',
       'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide',
       'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone',
       'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide',
       'examide', 'citoglipton', 'insulin', 'glyburide-metformin',
       'glipizide-metformin', 'glimepiride-pioglitazone',
       'metformin-rosiglitazone', 'metformin-pioglitazone'],
      dtype='object')


### 14. One Hot Encoding

Algorithms cannot understand text categories.

In [23]:
df = pd.get_dummies(
    df,
    columns=categorical_cols,
    drop_first=True
)

### 15.Feature Scaling

In [27]:
numerical_cols = [
    'time_in_hospital',
    'num_lab_procedures',
    'num_procedures',
    'num_medications',
    'number_outpatient',
    'number_emergency',
    'number_inpatient',
    'number_diagnoses',
    'total_visits',
    'total_procedures'
]


scaler = StandardScaler()

df[numerical_cols] = scaler.fit_transform(
    df[numerical_cols]
)

### 16. Saving Processed Dataset

In [37]:
import os

os.makedirs('/cleaned_data', exist_ok=True)
df.to_csv(
    "/cleaned_data/diabetic_processed.csv",
    index=False
)